# 12 · Architectures & Training — Beating Spectral Bias

**Level:** Research
**Prerequisites:** Beginner–Advanced track

Across the whole series one villain kept reappearing: **spectral bias**, the
tendency of plain MLPs to learn low frequencies first and high frequencies very
slowly (or never). This notebook demonstrates the problem cleanly and then fixes
it with **Fourier feature embeddings**, the single most effective architectural
upgrade for PINNs. We also summarise the modern training toolkit —
modified-MLP, self-adaptive loss weights, causal training, and PirateNets — that
defines the current state of the art.

### What you will learn
1. A reproducible demonstration of spectral bias on a high-frequency PDE
2. Random Fourier features (Tancik et al. 2020; Wang et al. 2021) and why they help
3. A map of modern PINN training techniques and where to read more

### References
- Tancik et al. (2020), *Fourier features let networks learn high-frequency functions*, NeurIPS
- Wang, Wang & Perdikaris (2021), *On the eigenvector bias of Fourier feature networks*, CMAME
- Wang, Sankaran, Wang & Perdikaris (2023), *An expert's guide to training PINNs*, CMAME
- Wang et al. (2024), *PirateNets: Physics-informed deep learning with residual adaptive networks*


## 1 · A deliberately hard target

Solve the 1-D Poisson equation

$$u''(x) = f(x),\quad x\in[0,1],\quad u(0)=u(1)=0,$$

with $f$ chosen so the exact solution is high-frequency:

$$u^\star(x)=\sin(6\pi x)\;\Rightarrow\; f(x)=-(6\pi)^2\sin(6\pi x).$$

A plain tanh-MLP struggles badly here; the Fourier-feature version nails it.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k = 6*np.pi
def exact(x): return np.sin(k*x)
def f_rhs(x): return -(k**2)*torch.sin(k*x)

xf = torch.linspace(0,1,400,device=device).view(-1,1).requires_grad_(True)
xb = torch.tensor([[0.0],[1.0]],device=device)


In [ ]:
class PlainMLP(nn.Module):
    def __init__(self, w=64, d=4):
        super().__init__()
        layers=[1]+[w]*d+[1]; seq=[]
        for i in range(len(layers)-1):
            seq.append(nn.Linear(layers[i],layers[i+1]))
            if i<len(layers)-2: seq.append(nn.Tanh())
        self.net=nn.Sequential(*seq)
    def forward(self,x): return self.net(x)

class FourierMLP(nn.Module):
    # random Fourier features: x -> [cos(Bx), sin(Bx)] then MLP
    def __init__(self, w=64, d=4, n_feat=64, sigma=6.0):
        super().__init__()
        B = torch.randn(1, n_feat, device=device)*sigma
        self.register_buffer("B", B)
        layers=[2*n_feat]+[w]*d+[1]; seq=[]
        for i in range(len(layers)-1):
            seq.append(nn.Linear(layers[i],layers[i+1]))
            if i<len(layers)-2: seq.append(nn.Tanh())
        self.net=nn.Sequential(*seq)
    def features(self,x):
        proj = 2*np.pi*x @ self.B
        return torch.cat([torch.cos(proj), torch.sin(proj)],dim=1)
    def forward(self,x): return self.net(self.features(x))


In [ ]:
def train(model, epochs=8000, lr=1e-3):
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    for ep in range(epochs):
        opt.zero_grad()
        u=model(xf)
        u_x=torch.autograd.grad(u,xf,torch.ones_like(u),create_graph=True)[0]
        u_xx=torch.autograd.grad(u_x,xf,torch.ones_like(u_x),create_graph=True)[0]
        L=torch.mean((u_xx-f_rhs(xf))**2)+50*torch.mean(model(xb)**2)
        L.backward(); opt.step()
    return model

def evaluate(model):
    xs=np.linspace(0,1,400)
    xt=torch.tensor(xs,dtype=torch.float32,device=device).view(-1,1)
    with torch.no_grad(): up=model(xt).cpu().numpy().ravel()
    ue=exact(xs); return np.linalg.norm(up-ue)/np.linalg.norm(ue), xs, up, ue


In [ ]:
m_plain = train(PlainMLP().to(device))
e_plain,xs,up_plain,ue = evaluate(m_plain)
print(f"[plain MLP]     rel L2 = {e_plain:.3e}")

m_fourier = train(FourierMLP().to(device))
e_four,_,up_four,_ = evaluate(m_fourier)
print(f"[Fourier feats] rel L2 = {e_four:.3e}")


In [ ]:
plt.figure(figsize=(9,4))
plt.plot(xs,ue,'k-',lw=2,label='exact  sin(6πx)')
plt.plot(xs,up_plain,'b--',lw=2,label=f'plain MLP (L2={e_plain:.1e})')
plt.plot(xs,up_four,'r-.',lw=2,label=f'Fourier features (L2={e_four:.1e})')
plt.legend(); plt.xlabel('x'); plt.ylabel('u'); plt.title('Fourier features defeat spectral bias')
plt.tight_layout(); plt.show()


## 2 · The modern PINN training toolkit

Fourier features are one piece. The techniques below, largely from the
Perdikaris / Karniadakis groups, together define current best practice
(see Wang et al. 2023, *An expert's guide to training PINNs*):

| Technique | Problem it solves | Key reference |
|---|---|---|
| **Adam → L-BFGS** two-stage | last orders of magnitude of accuracy | standard practice |
| **Fourier features** | spectral bias / high frequencies | Tancik 2020; Wang 2021 |
| **Modified MLP** (gating between two encoders) | gradient pathologies | Wang, Teng & Perdikaris 2021 |
| **Self-adaptive / NTK loss weights** | imbalanced loss terms | Wang, Yu & Perdikaris 2022 |
| **Causal training** | violating temporal causality | Wang, Sankaran & Perdikaris 2022 |
| **Hard-constraint ansatz** | exact BC/IC satisfaction | Lu et al. 2021 |
| **Domain decomposition** (XPINN/cPINN/FBPINN) | large / multiscale domains | Jagtap & Karniadakis 2020-21 |
| **RAR / RAD adaptive sampling** | sharp features | Lu 2021; Wu 2023 |
| **PirateNets** (residual adaptive) | training very deep PINNs stably | Wang et al. 2024 |
| **PIKANs** (KAN backbone) | interpretability, accuracy | Liu 2024; Toscano 2025 |

**PirateNets** (in your references folder) is a good capstone read: it uses
physics-informed initialisation plus adaptive residual connections so that very
deep networks — normally unstable for PINNs — train reliably and set
state-of-the-art accuracy on several benchmarks.